In [ ]:
import pandas as pd
import random
import subprocess
import re


In [ ]:

def load_steps_from_row(row, prefix="step-"):
    """
    Extracts non-empty steps from row
    """
    steps = []
    for col in sorted(row.index):
        if col.startswith(prefix):
            cell = str(row[col]).strip()
            # Skip if empty or just a placeholder (like NaN)
            if cell and cell.lower() != "nan":
                steps.append(cell)
    return steps


In [ ]:

def generate_distractor(correct_steps, num_distractors=3):
    """
    Generates list of distractor sequences by shuffling the correct sequence
    """
    distractors = []
    for _ in range(num_distractors):
        shuffled = correct_steps.copy()
        # Reshuffle until it is not the same as the correct order.
        attempt = 0
        while True:
            random.shuffle(shuffled)
            attempt += 1
            if shuffled != correct_steps or attempt > 10:
                break
        distractors.append("; ".join(shuffled))
    return distractors


In [ ]:

def build_prompt(recipe_name, correct_seq, distractors):
    """
    Builds prompt for given
    """
    # Format the sequences as a string.
    correct_option = "; ".join(correct_seq)
    options = [{"text": correct_option, "is_correct": True}]
    for d in distractors:
        options.append({"text": d, "is_correct": False})

    # Randomize option order
    random.shuffle(options)

    # Map letters (a, b, c, d) to the options.
    letters = ['a', 'b', 'c', 'd']
    option_lines = []
    correct_letter = None
    for i, option in enumerate(options):
        letter = letters[i]
        option_lines.append(f"({letter}) {option['text']}")
        if option["is_correct"]:
            correct_letter = letter

    prompt = f"""Identify the correct sequence of steps for this recipe from the options below:

Task: {recipe_name}
Options:
{chr(10).join(option_lines)}
"""
    return prompt, correct_letter

def query_llm(model, prompt):
    """
    Calls gemma3 model via Ollama
    """
    try:
        result = subprocess.run(
            ["ollama", "run", model, prompt],
            capture_output=True,
            text=True,
            check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print("Error querying model:", e)
        return ""


In [ ]:

def extract_choice(response):
    """
    Extracts option letter from the LLM response.
    """
    response = response.lower()
    match = re.search(r'\b([abcd])\b', response)
    if match:
        return match.group(1)
    return None


In [ ]:
def evaluate_model(ground_truth_csv, jumbled_csv, model="gemma3"):
    # Load the ground truth and jumbled data
    df_gt = pd.read_csv(ground_truth_csv)
    df_jumbled = pd.read_csv(jumbled_csv)

    total = 0
    correct_count = 0
    results = []  # store details for each example

    # Assuming both CSVs have a column "Name" to join on.
    for index, gt_row in df_gt.iterrows():
        recipe_name = gt_row["Name"]
        # Get the correct steps from the ground truth row (assume columns "step-1", "step-2", ...)
        correct_steps = load_steps_from_row(gt_row)
        if not correct_steps:
            continue

        # Find the corresponding row in jumbled CSV
        jumbled_row = df_jumbled[df_jumbled["Name"] == recipe_name]
        if jumbled_row.empty:
            continue
        jumbled_steps = load_steps_from_row(jumbled_row.iloc[0])
        # For distractors, you can use the jumbled_steps if available or simply generate shuffles from the correct_steps.
        # Here, we generate distractors by shuffling the correct order.
        distractors = generate_distractor(correct_steps, num_distractors=3)

        # Build prompt and record which option letter is the correct one.
        prompt, correct_letter = build_prompt(recipe_name, correct_steps, distractors)

        # Query the gemma3 model using Ollama.
        llm_response = query_llm(model, prompt)
        chosen_letter = extract_choice(llm_response)

        # Determine if the model's answer is correct.
        is_correct = (chosen_letter == correct_letter)
        correct_count += int(is_correct)
        total += 1

        results.append({
            "recipe": recipe_name,
            "prompt": prompt,
            "llm_response": llm_response,
            "extracted_choice": chosen_letter,
            "correct_choice": correct_letter,
            "is_correct": is_correct
        })
        # Optionally print progress every 100 recipes
        if total % 100 == 0:
            print(f"Processed {total} recipes so far...")

    # Calculate and print accuracy.
    accuracy = (correct_count / total) * 100 if total > 0 else 0
    print(f"\nEvaluated {total} recipes.")
    print(f"Model accuracy on control flow evaluation: {accuracy:.2f}%")
    return results


In [ ]:

if __name__ == "__main__":
    # File paths for the correct and jumbled recipes
    ground_truth_csv = "recipes_split.csv"       # Correct sequence file
    jumbled_csv = "controlflow_recipies.csv"       # Jumbled steps file (from [1])

    # Evaluate gemma3 (you can later extend or loop over other models such as llama3.3, deepseek-r1, phi4, mistral)
    evaluation_results = evaluate_model(ground_truth_csv, jumbled_csv, model="gemma3")
